In [ ]:
import sys
import os
import logging
logging.basicConfig(level=logging.INFO)
# Add the project root to Python path
# project_root = os.path.dirname(os.getcwd())
# sys.path.insert(0, project_root)

sys.path

['/home/abhishek/snap/code/205/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python311.zip',
 '/home/abhishek/snap/code/205/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11',
 '/home/abhishek/snap/code/205/.local/share/uv/python/cpython-3.11.13-linux-x86_64-gnu/lib/python3.11/lib-dynload',
 '',
 '/home/abhishek/Downloads/work/active_gliner/.venv/lib/python3.11/site-packages']

In [2]:
os.path.dirname(os.getcwd())

'/home/abhishek/Downloads/work/active_gliner'

In [3]:
sys.path.append(os.path.dirname(os.getcwd()))

In [4]:
from src.experiment import active_gliner as ag

/home/abhishek/Downloads/work/active_gliner/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
cfg = ag.load_config()
cfg

{'io': {'data_dir': '../data/mit-movie',
  'results_dir': 'results',
  'results_filename': 'results.json',
  'logs_dir': 'logs',
  'models_dir': 'models'},
 'model': {'base_model': 'knowledgator/modern-gliner-bi-large-v1.0',
  'max_len': 8192,
  'seed': 42},
 'lora': {'enabled': True,
  'r': 32,
  'lora_alpha': 64,
  'lora_dropout': 0.1,
  'bias': 'none',
  'task_type': 'TOKEN_CLS',
  'target_modules': ['dense',
   'projection',
   'Wqkv',
   'Wo',
   'Wi',
   'query',
   'key',
   'value',
   'intermediate.dense',
   'output.dense',
   'span_rep_layer.span_rep_layer.project_start.3',
   'span_rep_layer.span_rep_layer.project_start.0',
   'span_rep_layer.span_rep_layer.project_end.3',
   'span_rep_layer.span_rep_layer.project_end.0',
   'span_rep_layer.span_rep_layer.out_project.3',
   'span_rep_layer.span_rep_layer.out_project.0',
   'prompt_rep_layer.3',
   'prompt_rep_layer.0']},
 'training': {'num_steps': 200,
  'per_device_train_batch_size': 8,
  'per_device_eval_batch_size': 8,
 

In [6]:
ag.ensure_dirs(cfg)

In [7]:
train, test, labels = ag.load_dataset(cfg)

In [8]:
results = ag.compute_train_subset_results(cfg, train, labels)

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 71765.66it/s]


Running enhanced evaluation...
Processing 2932 examples...


Could not save results to results/results.json: Object of type Styler is not JSON serializable


Analyzing errors with ground truth...


In [9]:
results

{'overall_metrics': {'total_predictions': 3291,
  'overall_confidence': np.float64(0.7860185189364494),
  'overall_confidence_pct': np.float64(78.60185189364493),
  'total_examples': 2932,
  'entity_level_accuracy': np.float64(0.37434390090279235),
  'entity_level_accuracy_pct': np.float64(37.43439009027924),
  'example_level_accuracy': 0.2121418826739427,
  'example_level_accuracy_pct': 21.21418826739427,
  'overall_f1': np.float64(0.44276136081450207),
  'overall_f1_pct': np.float64(44.276136081450204),
  'incorrect_examples': [{'tokenized_text': ['show',
     'me',
     'films',
     'with',
     'drew',
     'barrymore',
     'from',
     'the',
     '1980s'],
    'ner': [(4, 5, 'actor'), (8, 8, 'year')],
    'predictions': [],
    'scores': [],
    'errors': {'false_negatives': [[8, 8, 'year'], [4, 5, 'actor']],
     'false_positives': []}},
   {'tokenized_text': ['find',
     'me',
     'a',
     'movie',
     'with',
     'a',
     'quote',
     'about',
     'baseball',
     'i

In [10]:
low = ag.select_low_confidence(results, n=cfg['experiment']['num_corrected'])

In [11]:
analysis = ag.get_or_create_analysis(cfg, low, labels)
print(analysis)

final_summary = analysis.get('final_summary') if analysis else None
print(final_summary)
synthetic = ag.get_or_create_synthetic(cfg, low, labels, final_summary)
print((synthetic))

{'final_summary': {'domain_summary': 'The system performs well in identifying genres, actors, titles, and reviews. However, there are issues with the identification of year, plot, character, rating, and trailer entities. Key variations needed for year, plot, character, rating, and trailer.', 'entity_summaries': {'genre': {'position_summary': 'Correctly identified when it appears as a noun following a verb', 'good_examples_summary': 'can give give me the critically acclaimed comedies made in the 1960s', 'bad_examples_summary': '', 'variations_summary': ''}, 'year': {'position_summary': "Correctly identified when it appears as a numeric value following 'made in' or 'from'", 'good_examples_summary': 'can give give me the critically acclaimed comedies made in the 1960s', 'bad_examples_summary': 'was there a time travelling astronaut film', 'variations_summary': "Consider handling variations where the year is not explicitly stated as 'made in' or 'from'"}, 'plot': {'position_summary': 'Corr

Generating synthetic data: 100%|██████████| 20/20 [03:55<00:00, 11.78s/it]

[{'tokenized_text': ['I', "'", 'd', 'like', 'to', 'watch', 'a', 'suspenseful', 'British', 'horror', 'film', 'from', 'the', 'late', '1970s', ',', 'featuring', 'an', 'iconic', 'performance', 'by', 'Sir', 'Michael', 'Caine', 'as', 'a', 'haunted', 'antiques', 'dealer', '.', 'Do', 'you', 'have', 'any', 'recommendations', 'for', 'such', 'a', 'movie', ',', 'and', 'can', 'you', 'also', 'provide', 'a', 'link', 'to', 'its', 'official', 'trailer', '?'], 'ner': [(7, 10, 'genre'), (7, 10, 'title'), (13, 14, 'year'), (18, 23, 'actor'), (26, 28, 'character'), (34, 34, 'review'), (49, 50, 'trailer')]}, {'tokenized_text': ['I', "'", 'd', 'like', 'to', 'watch', 'a', 'suspenseful', 'psychological', 'thriller', 'from', 'the', '2010s', ',', 'featuring', 'Leonardo', 'DiCaprio', 'as', 'the', 'protagonist', '.', 'Could', 'you', 'please', 'provide', 'me', 'with', 'the', 'trailer', 'and', 'the', 'average', 'ratings', 'for', 'this', 'movie', '?'], 'ner': [(7, 9, 'genre'), (12, 12, 'year'), (15, 16, 'actor'), (19

In [12]:
print(len(synthetic))

19


In [13]:
split=len(synthetic)*.80
train_data=synthetic[:int(split)]
val_data=synthetic[int(split):]

print("before corrected labels")
print(len(train_data), len(val_data))
print("after corrected labels")
train_data.extend(low)
print(len(train_data), len(val_data))

before corrected labels
15 4
after corrected labels
20 4


In [14]:
model, monitor = ag.train_with_synthetic(cfg, train_data, val_data)

Fetching 9 files: 100%|██████████| 9/9 [00:00<00:00, 140853.49it/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50283}.


Step,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
test_results = ag.evaluate_on_test(cfg, model, test, labels, device)

NameError: name 'model' is not defined